# 第 8 讲：Parallelism Basics

这是 CS336 Lecture 08 的可执行中文学习版。

- 官方来源：`external/cs336-lectures/lecture_08.pdf`
- 官方形式：PDF lecture
- 英文标题：Parallelism Basics

这份 notebook 是学习版，不是逐字翻译。它按官方讲义的结构和节奏整理主线，并在适合的位置加入小实验。


## 0. 本讲你要抓住什么

这一讲从网络和集群视角补充分布式训练：为什么多种并行方式经常组合使用，以及大规模训练 run 通常长什么样。

学习方式：

1. 先读每节的中文解释。
2. 运行对应代码 cell。
3. 改一个参数，观察输出如何变化。
4. 最后用检查点问题自测。


## 1. 本讲路线图

网络基础、并行策略、组合并行、大规模训练实践。


In [1]:
lecture = 8
title = 'Parallelism Basics'
source = 'external/cs336-lectures/lecture_08.pdf'
print(f"Lecture {lecture}: {title}")
print("source:", source)


Lecture 8: Parallelism Basics
source: external/cs336-lectures/lecture_08.pdf


## 2. 网络不是免费的

跨 GPU/节点通信有带宽和延迟；通信模式会限制扩展效率。


## 3. 为什么组合并行

data parallel 解决吞吐，tensor parallel 解决单层太大，pipeline/FSDP 解决显存。


## 4. FSDP / ZeRO 直觉

把参数、梯度、optimizer state 分片，必要时 all-gather，用完释放。


## 5. 大 run 的现实

训练不是只启动一次脚本，还包括 checkpoint、故障恢复、监控、数据吞吐和性能回归。


## 动手实验 1：ZeRO 分片显存粗算

先直接运行，再改一个数字或字符串。


In [2]:
params = 70e9
bytes_per_param_state = 2 + 2 + 4 + 4  # param, grad, Adam moments
for world in [1, 8, 64, 512]:
    gb = params * bytes_per_param_state / world / 1e9
    print(f"world={world:3d} per-rank state={gb:.1f} GB")


world=  1 per-rank state=840.0 GB
world=  8 per-rank state=105.0 GB
world= 64 per-rank state=13.1 GB
world=512 per-rank state=1.6 GB


## 动手实验 2：通信时间粗算

先直接运行，再改一个数字或字符串。


In [3]:
def transfer_time_gb(size_gb, bandwidth_gb_s, latency_ms=0.02):
    return latency_ms / 1000 + size_gb / bandwidth_gb_s

for size in [0.1, 1, 10]:
    print(size, "GB over 400GB/s:", transfer_time_gb(size, 400), "s")


0.1 GB over 400GB/s: 0.00027 s
1 GB over 400GB/s: 0.00252 s
10 GB over 400GB/s: 0.02502 s


## 今日检查点

1. 能解释为什么通信带宽会限制扩展。
2. 能说出 FSDP/ZeRO 通过分片省了什么。
3. 能理解大规模训练需要 checkpoint 和故障恢复。

如果这些能讲清楚，这一讲的第一轮学习就完成了。
